# CSE Policy Impact Predictor — University Research Platform
## Notebook 00: Environment Setup and Dataset Inventory Verification

**Project Title:** Predicting Stock Market Movements Based on Government Policy Events Using Machine Learning  
**Module:** Government Policy Stock Market Prediction  
**Phase:** Phase 0 — Project Inspection and Initial Setup  

### Objective
This notebook verifies the Google Colab environment, loads the raw Colombo Stock Exchange (CSE) datasets and the 70-event Government Policy dataset, validates data integrity, and outputs dataset inventory metrics without modifying raw data.

### 1. Environment and Hardware Verification
Confirm Python version, CPU/GPU availability, and working directories.

In [ ]:
import sys
import os
import platform

print(f"Python Version: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Current Directory: {os.getcwd()}")

### 2. Dependency Installation
Install and import required packages: `pandas`, `numpy`, `openpyxl`, `xlrd`, `scikit-learn`, `matplotlib`, `seaborn`.

In [ ]:
# Install dependencies if running on Google Colab
!pip install -q pandas numpy openpyxl xlrd scikit-learn matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

### 3. File Path Configuration
Support both Google Drive mounts (`/content/drive/MyDrive/...`) and local directory execution.

In [ ]:
# Check for Google Drive mount
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Set base directory to your drive folder if using Colab
    BASE_DIR = '/content/drive/MyDrive/research-dev'
except Exception:
    # Fallback to local workspace or current directory
    BASE_DIR = os.getcwd()

POLICY_FILE = os.path.join(BASE_DIR, 'Dataset-Policy', 'Government_Policies.csv')
MARKET_INDICES_FILE = os.path.join(BASE_DIR, 'Dataset-Stock', '07Market Indices - Daily.xls')
MARKET_STATS_FILE = os.path.join(BASE_DIR, 'Dataset-Stock', '11Market Statistics-Daily.xls')

print(f"Using BASE_DIR: {BASE_DIR}")
print(f"Policy file exists: {os.path.exists(POLICY_FILE)}")
print(f"Indices file exists: {os.path.exists(MARKET_INDICES_FILE)}")
print(f"Stats file exists: {os.path.exists(MARKET_STATS_FILE)}")

### 4. Government Policy Dataset Inspection
Inspect structure, row count, column dimensions, date ranges, and category distributions for the 70 policy events.

In [ ]:
if os.path.exists(POLICY_FILE):
    df_policy = pd.read_csv(POLICY_FILE)
    print(f"Policy Dataset Shape: {df_policy.shape[0]} rows x {df_policy.shape[1]} columns\n")
    print("Columns:", list(df_policy.columns))
    print("\nMissing values per column:")
    print(df_policy.isnull().sum()[df_policy.isnull().sum() > 0])
    
    print("\nPolicy Types Distribution:")
    print(df_policy['Policy_Type'].value_counts())
    
    print("\nDate Ranges:")
    for col in ['Primary_Event_Date', 'Announcement_Date', 'Approval_Date', 'Effective_Date', 'CSE_Trading_Day']:
        if col in df_policy.columns:
            dates = pd.to_datetime(df_policy[col], errors='coerce').dropna()
            print(f"  {col}: {dates.min().strftime('%Y-%m-%d')} to {dates.max().strftime('%Y-%m-%d')}")
else:
    print("ERROR: Policy file not found!")

### 5. Colombo Stock Exchange (CSE) Market Indices Inspection
Examine historical All Share Price Index (ASPI), S&P Sri Lanka 20, and sector indices from `07Market Indices - Daily.xls`.

In [ ]:
if os.path.exists(MARKET_INDICES_FILE):
    df_raw = pd.read_excel(MARKET_INDICES_FILE, sheet_name='Index', header=None)
    print(f"Market Indices Sheet Shape: {df_raw.shape}")
    
    # Parse dates and index values
    dates = pd.to_datetime(df_raw.iloc[5:, 0], errors='coerce')
    aspi = pd.to_numeric(df_raw.iloc[5:, 1], errors='coerce')
    sp20 = pd.to_numeric(df_raw.iloc[5:, 3], errors='coerce')
    
    df_market = pd.DataFrame({'Date': dates, 'ASPI': aspi, 'SP_SL20': sp20}).dropna(subset=['Date'])
    df_market = df_market.sort_values('Date').reset_index(drop=True)
    
    print(f"Total Trading Days: {len(df_market):,}")
    print(f"Historical Date Range: {df_market['Date'].min().strftime('%Y-%m-%d')} to {df_market['Date'].max().strftime('%Y-%m-%d')}")
    print(f"ASPI Valid Count: {df_market['ASPI'].notnull().sum():,}, Latest: {df_market['ASPI'].dropna().iloc[-1]:.2f}")
    print(f"S&P SL20 Valid Count: {df_market['SP_SL20'].notnull().sum():,}, Latest: {df_market['SP_SL20'].dropna().iloc[-1]:.2f}")
    
    # Focus period 2020-2025
    focus_df = df_market[(df_market['Date'] >= '2020-01-01') & (df_market['Date'] <= '2025-12-31')]
    print(f"\nPrimary Focus (2020-2025) Trading Days: {len(focus_df):,}")
else:
    print("ERROR: Market indices file not found!")

### 6. Preliminary Market Visual Check (2020–2025 ASPI)
Plot the historical All Share Price Index over the primary research period.

In [ ]:
if 'focus_df' in locals() and len(focus_df) > 0:
    plt.figure(figsize=(12, 5), dpi=100)
    plt.plot(focus_df['Date'], focus_df['ASPI'], color='#059669', linewidth=1.5, label='CSE ASPI Index')
    plt.title('Colombo Stock Exchange — All Share Price Index (ASPI) 2020–2025', fontsize=12, fontweight='bold')
    plt.xlabel('Date')
    plt.ylabel('Index Value')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.savefig('aspi_2020_2025_preview.png')
    plt.show()
    print("Saved preview chart: aspi_2020_2025_preview.png")

### 7. Export Dataset Compatibility Summary
Generate metadata JSON verifying all Phase 0 dataset prerequisites are satisfied.

In [ ]:
import json

summary = {
    "phase": "Phase 0 — Project Inspection and Initial Setup",
    "status": "VERIFIED",
    "policy_dataset": {
        "file": os.path.basename(POLICY_FILE),
        "records": int(len(df_policy)) if 'df_policy' in locals() else 0,
        "columns": int(df_policy.shape[1]) if 'df_policy' in locals() else 0
    },
    "stock_dataset": {
        "file": os.path.basename(MARKET_INDICES_FILE),
        "total_trading_days": int(len(df_market)) if 'df_market' in locals() else 0,
        "focus_period_trading_days": int(len(focus_df)) if 'focus_df' in locals() else 0,
        "date_range": f"{df_market['Date'].min().strftime('%Y-%m-%d')} to {df_market['Date'].max().strftime('%Y-%m-%d')}" if 'df_market' in locals() else "N/A"
    },
    "environment_compatible": True
}

with open('dataset_inventory_colab.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("Compatibility report generated: dataset_inventory_colab.json")
print(json.dumps(summary, indent=2))